# Capstone Project 1 — Part 1 (Google Colab): EDA, Model Selection & Tuning
**Track:** Option A — Online Fraud Transaction Detection · **Author:** Abhimanyu Kumar Singh

This notebook covers Section 1 of the brief up to choosing the best model and its hyperparameters.
It writes a structured `artifacts/` folder to Google Drive. The VS Code project (Part 2) reads that folder
to build the FastAPI + CI + Azure MLflow application.

**Before running**
1. Google Drive: the dataset is at `MyDrive/fraud-detection/data/onlinefraud.csv`.
   Add a `src/` subfolder to `MyDrive/fraud-detection/` with `__init__.py` and `features.py` copied from the
   VS Code project (the same file must be used on both sides). Results go to `MyDrive/fraud-detection/artifacts/`.
2. Runtime → Change runtime type → CPU is enough (High-RAM helps if you have it).
3. Run all. With `BUDGET = "quick"` the whole notebook takes roughly 20–30 minutes on a standard Colab CPU.

**Objective: maximise F1** on the fraud class. F1 drives the imbalance-strategy choice, the hyperparameter search
(`refit="f1"`), the decision threshold (F1-optimal on validation) and the final model selection (validation F1).
   Every search is checkpointed to Drive, so after a disconnect just re-run all: finished models are skipped.

In [ ]:
# 0.1 Install / import (most packages are preinstalled on Colab)
!pip -q install imbalanced-learn xgboost shap

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 0.2 Configuration
import json, os, platform, shutil, sys, time, warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_DIR = Path("/content/drive/MyDrive/fraud-detection")   # Drive folder that already holds the dataset

DATA_FILE = PROJECT_DIR / "data" / "onlinefraud.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"
ART = PROJECT_DIR / "artifacts"                         # everything Part 2 needs goes here
FIG = ART / "figures"
CKPT = ART / "checkpoints"
for d in (ART, FIG, CKPT, ART / "cv_results", ART / "models"):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))
from src.features import (  # noqa: E402  (shared module, identical in the VS Code repo)
    ENGINEERED_FEATURES, ID_COLUMNS, INPUT_FEATURES, LEAKY_COLUMNS, MONEY_FEATURES,
    RAW_CATEGORICAL, RAW_NUMERIC, TARGET, TYPE_CATEGORIES,
    FraudFeatureEngineer, build_preprocessor, standardize_columns,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Compute budget. "quick" keeps the total run time short; "standard" searches more widely.
BUDGET = "quick"
BUDGETS = {
    "quick": dict(neg_pos_ratio=20, cv_folds=3, imb_sample=60_000, rf_max_trees=120,
                  n_iter={"logreg": 6, "dtree": 8, "rf": 4, "xgb": 8, "hgb": 6}),
    "standard": dict(neg_pos_ratio=50, cv_folds=3, imb_sample=150_000, rf_max_trees=300,
                     n_iter={"logreg": 10, "dtree": 15, "rf": 8, "xgb": 15, "hgb": 10}),
}
CFG = BUDGETS[BUDGET]
RESUME = True          # skip models whose checkpoint already exists in Drive
SAMPLER_RATIO = 0.5    # minority/majority ratio after any resampling step

# Optimisation target: F1 of the fraud class (search refit, threshold, model selection).
OBJECTIVE = "f1"
# Final model chosen for deployment. The criterion-based pick (validation F1, ties -> CV F1) is still computed and
# recorded; set FINAL_MODEL = None to deploy that pick instead.
FINAL_MODEL = "rf"   # Random Forest
# Business cost, reported alongside F1 for the FN-vs-FP discussion in the report (not optimised):
# a missed fraud loses the whole transaction amount, a false alarm costs a review + customer friction.
COST_FN, COST_FP = 25, 1

TIMINGS = {}
def tic(name):
    TIMINGS[name] = time.time()
def toc(name):
    TIMINGS[name] = round(time.time() - TIMINGS[name], 1)
    print(f"[{name}] {TIMINGS[name]}s")

def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.ndarray):
        return to_jsonable(o.tolist())
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, Path):
        return str(o)
    return o

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)
    print("saved", path.relative_to(PROJECT_DIR))

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png", dpi=120, bbox_inches="tight")
    plt.show()

print("Budget:", BUDGET, CFG)

## 1. Data loading and inspection

In [ ]:
tic("load")
LOCAL = Path("/content/data"); LOCAL.mkdir(exist_ok=True)
csv_path = LOCAL / DATA_FILE.name
if not csv_path.exists():                      # local copy reads much faster than the Drive mount
    shutil.copy(DATA_FILE, csv_path)

dtypes = {"step": "int16", "type": "category", "amount": "float64",
          "nameOrig": "string", "oldbalanceOrg": "float64", "newbalanceOrig": "float64",
          "nameDest": "string", "oldbalanceDest": "float64", "newbalanceDest": "float64",
          "isFraud": "int8", "isFlaggedFraud": "int8"}
df = pd.read_csv(csv_path, dtype=dtypes)
toc("load")

expected = set(dtypes)
assert expected.issubset(df.columns), f"Unexpected schema. Missing: {expected - set(df.columns)}"
print("Shape:", df.shape)
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()

In [ ]:
display(df.dtypes.to_frame("dtype"))
display(df.describe().T)
display(df.describe(include=["category", "string"]).T)
missing = df.isna().sum()
print("Missing values per column:\n", missing)

## 2. Data cleaning
Every decision is logged to `artifacts/cleaning_log.json` for the report.

In [ ]:
tic("clean")
clean_log = {"rows_raw": len(df), "decisions": []}

clean_log["missing_per_column"] = missing.to_dict()
if missing.sum() > 0:
    df = df.dropna()
    clean_log["decisions"].append(f"Dropped {int(missing.sum())} rows with missing values.")
else:
    clean_log["decisions"].append("No missing values found; no imputation needed.")

n_dup = int(df.duplicated().sum())
df = df.drop_duplicates()
clean_log["duplicates_removed"] = n_dup
clean_log["decisions"].append(f"Removed {n_dup} exact duplicate rows.")

df = standardize_columns(df)

# Unknown categories / negative amounts
bad_type = int((~df["type"].isin(TYPE_CATEGORIES)).sum())
neg_money = {c: int((df[c] < 0).sum()) for c in MONEY_FEATURES if c in df.columns}
clean_log["unknown_type_rows"] = bad_type
clean_log["negative_money_values"] = neg_money

# The system's own rule-based flag: check how it relates to the target, then drop (leakage / not a model input)
flag_tab = pd.crosstab(df["Is_Flagged_Fraud"], df[TARGET])
display(flag_tab)
clean_log["is_flagged_fraud_crosstab"] = flag_tab.to_dict()
clean_log["decisions"].append(
    "Dropped Is_Flagged_Fraud: a rule-based output of the source system (flags only TRANSFERs > 200k), "
    "derived from the outcome process; using it would leak and it is not available as an independent input.")

# Outliers: quantify with the IQR rule but KEEP them (extreme amounts are genuine fraud signal)
outlier_report = {}
for c in ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    hi = q3 + 1.5 * (q3 - q1)
    mask = df[c] > hi
    outlier_report[c] = {"iqr_upper_fence": float(hi), "n_outliers": int(mask.sum()),
                         "pct_outliers": float(mask.mean() * 100),
                         "fraud_rate_in_outliers_pct": float(df.loc[mask, TARGET].mean() * 100),
                         "fraud_rate_overall_pct": float(df[TARGET].mean() * 100)}
display(pd.DataFrame(outlier_report).T)
clean_log["outliers_iqr"] = outlier_report
clean_log["decisions"].append(
    "Outliers (IQR rule) kept: fraud rate inside the outlier band is higher than overall, so extreme "
    "values carry signal. Heavy tails are handled with a signed log transform inside the pipeline instead.")

# Identifier columns: quick facts for the report, then dropped
clean_log["unique_nameOrig_pct"] = float(df["nameOrig"].nunique() / len(df) * 100)
clean_log["unique_nameDest_pct"] = float(df["nameDest"].nunique() / len(df) * 100)
clean_log["merchant_dest_pct"] = float(df["nameDest"].str.startswith("M").mean() * 100)
df = df.drop(columns=ID_COLUMNS + LEAKY_COLUMNS)
clean_log["decisions"].append(
    "Dropped nameOrig / nameDest: identifiers (nameOrig is ~unique per row), so per-account velocity features "
    "are not reliable and would not be available to a real-time API without an account history store.")
clean_log["rows_clean"] = len(df)
save_json(clean_log, ART / "cleaning_log.json")
toc("clean")

## 3. Exploratory data analysis
Figures are saved to `artifacts/figures/` for the report.

In [ ]:
tic("eda")
feat = FraudFeatureEngineer().transform(df[INPUT_FEATURES])
eda = pd.concat([feat, df[[TARGET]]], axis=1)
eda["day"] = df["step"] // 24
eda["step"] = df["step"]
fraud = eda[eda[TARGET] == 1]
legit_s = eda[eda[TARGET] == 0].sample(100_000, random_state=RANDOM_STATE)
eda_s = pd.concat([fraud, legit_s])

summary = {}
vc = df[TARGET].value_counts()
summary["class_counts"] = vc.to_dict()
summary["fraud_rate_pct"] = float(df[TARGET].mean() * 100)
summary["imbalance_ratio_legit_per_fraud"] = float(vc[0] / vc[1])

fig, ax = plt.subplots(figsize=(5, 3.5))
vc.plot.bar(ax=ax, logy=True, color=["#4C72B0", "#C44E52"])
ax.set_title(f"Class balance (fraud = {summary['fraud_rate_pct']:.3f}%)"); ax.set_xlabel(TARGET); ax.set_ylabel("count (log)")
savefig("01_class_balance")
print(summary)

In [ ]:
by_type = df.groupby("type", observed=True)[TARGET].agg(["count", "sum", "mean"]).rename(
    columns={"count": "transactions", "sum": "frauds", "mean": "fraud_rate"})
display(by_type)
summary["by_type"] = by_type.to_dict(orient="index")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
by_type["transactions"].plot.bar(ax=axes[0], color="#4C72B0"); axes[0].set_title("Transactions by type")
by_type["frauds"].plot.bar(ax=axes[1], color="#C44E52"); axes[1].set_title("Frauds by type")
savefig("02_type")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for lbl, g, col in [(0, legit_s, "#4C72B0"), (1, fraud, "#C44E52")]:
    sns.kdeplot(np.log10(g["amount"] + 1), ax=ax, label=f"{TARGET}={lbl}", fill=True, alpha=0.3, color=col)
ax.set_xlabel("log10(amount + 1)"); ax.set_title("Amount distribution by class"); ax.legend()
savefig("03_amount_by_class")
summary["amount_median_by_class"] = eda.groupby(TARGET)["amount"].median().to_dict()
summary["amount_mean_by_class"] = eda.groupby(TARGET)["amount"].mean().to_dict()

In [ ]:
hourly = eda.groupby("hour")[TARGET].agg(["count", "mean"])
fig, ax1 = plt.subplots(figsize=(9, 3.8))
ax1.bar(hourly.index, hourly["count"], color="#B0C4DE", label="transactions")
ax1.set_xlabel("hour of day (step % 24)"); ax1.set_ylabel("transactions")
ax2 = ax1.twinx(); ax2.plot(hourly.index, hourly["mean"] * 100, color="#C44E52", marker="o", label="fraud rate %")
ax2.set_ylabel("fraud rate %"); ax1.set_title("Volume and fraud rate by hour of day")
savefig("04_hour_of_day")
summary["fraud_rate_by_hour_pct"] = (hourly["mean"] * 100).round(4).to_dict()
summary["night_fraud_rate_pct"] = float(eda.loc[eda.is_night == 1, TARGET].mean() * 100)
summary["day_fraud_rate_pct"] = float(eda.loc[eda.is_night == 0, TARGET].mean() * 100)

In [ ]:
daily = eda.groupby("day")[TARGET].agg(["count", "sum"])
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(daily.index, daily["count"], label="transactions"); ax.set_ylabel("transactions"); ax.set_xlabel("day (step // 24)")
ax2 = ax.twinx(); ax2.plot(daily.index, daily["sum"], color="#C44E52", label="frauds"); ax2.set_ylabel("frauds")
ax.set_title("Transactions and frauds over time")
savefig("05_over_time")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, col in zip(axes, ["error_balance_orig", "error_balance_dest"]):
    for lbl, g, c in [(0, legit_s, "#4C72B0"), (1, fraud, "#C44E52")]:
        v = np.sign(g[col]) * np.log1p(np.abs(g[col]))
        ax.hist(v, bins=80, alpha=0.5, density=True, label=f"{TARGET}={lbl}", color=c)
    ax.set_title(f"{col} (signed log)"); ax.legend()
savefig("06_balance_errors")
summary["full_drain_rate_by_class_pct"] = (eda.groupby(TARGET)["is_full_drain"].mean() * 100).to_dict()
summary["orig_zero_after_rate_by_class_pct"] = (eda.groupby(TARGET)["orig_zero_after"].mean() * 100).to_dict()

In [ ]:
num_cols = [c for c in RAW_NUMERIC + ENGINEERED_FEATURES if c in eda_s.columns]
corr = eda_s[num_cols + [TARGET]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, ax=ax)
ax.set_title("Spearman correlation (all frauds + 100k sampled legit)")
savefig("07_correlation")
summary["spearman_with_target"] = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(4).to_dict()
save_json(summary, ART / "eda_summary.json")
del feat, eda, eda_s, fraud, legit_s
toc("eda")

## 4. Train / validation / test split (time-based) and class-imbalance setup
- Data is time-ordered (`step` = hour), so the split is by time: first ~70% of rows → train, next ~15% → validation,
  last ~15% → test. No step is split across sets. (Validation was enlarged from 10% so that model selection and the
  threshold rest on more fraud cases.)
- **Validation** and **test** keep the true class distribution. Validation picks the model and the decision threshold;
  test is touched once for the final report.
- To keep the tuning time manageable, the **development sample** keeps *all* training frauds and a random sample of
  legitimate transactions (`neg_pos_ratio` legit per fraud). Stratified k-fold CV runs inside this sample, and any
  resampling (SMOTE etc.) happens **inside** the CV pipeline, so it never touches validation folds.

In [ ]:
tic("split")
df = df.sort_values("step", kind="stable").reset_index(drop=True)
n = len(df)
train_end_step = int(df["step"].iloc[int(n * 0.70)])
val_end_step = int(df["step"].iloc[int(n * 0.85)])
train = df[df["step"] <= train_end_step]
val = df[(df["step"] > train_end_step) & (df["step"] <= val_end_step)]
test = df[df["step"] > val_end_step]

split_info = {}
for name, part in [("train", train), ("val", val), ("test", test)]:
    split_info[name] = {"rows": len(part), "frauds": int(part[TARGET].sum()),
                        "fraud_rate_pct": float(part[TARGET].mean() * 100),
                        "step_min": int(part["step"].min()), "step_max": int(part["step"].max())}
    assert part[TARGET].sum() > 0, f"No frauds in {name}"
display(pd.DataFrame(split_info).T)

pos = train[train[TARGET] == 1]
neg = train[train[TARGET] == 0].sample(n=len(pos) * CFG["neg_pos_ratio"], random_state=RANDOM_STATE)
dev = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
X_dev, y_dev = dev[INPUT_FEATURES], dev[TARGET].astype(int)
X_val, y_val = val[INPUT_FEATURES], val[TARGET].astype(int)
X_test, y_test = test[INPUT_FEATURES], test[TARGET].astype(int)
split_info["dev_sample"] = {"rows": len(dev), "frauds": int(y_dev.sum()),
                            "fraud_rate_pct": float(y_dev.mean() * 100),
                            "neg_pos_ratio": CFG["neg_pos_ratio"],
                            "method": "all train frauds + random legit sample"}
split_info["boundaries"] = {"train_end_step": train_end_step, "val_end_step": val_end_step}
print(split_info["dev_sample"])

# Input ranges observed in training data (used for Pydantic validation in the API)
input_ranges = {c: {"min": float(train[c].min()), "max": float(train[c].max()),
                    "p99_9": float(train[c].quantile(0.999))} for c in RAW_NUMERIC}
input_ranges["step"]["max_overall"] = int(df["step"].max())
# Reference sample (true distribution, training period) for drift monitoring in Part 2
train.sample(20_000, random_state=RANDOM_STATE)[INPUT_FEATURES + [TARGET]].to_csv(ART / "reference_sample.csv", index=False)
del df, train, pos, neg
toc("split")

## 5. Class-imbalance strategies compared
Logistic Regression and a depth-limited Decision Tree are cross-validated on a stratified subsample of the dev set
with five strategies: none, class weights, random undersampling, random oversampling and SMOTE.
The winner (mean CV F1 across both models) is used for all models in Section 6.

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

SCORING = {"pr_auc": "average_precision", "roc_auc": "roc_auc", "f1": "f1",
           "recall": "recall", "precision": "precision"}
cv = StratifiedKFold(n_splits=CFG["cv_folds"], shuffle=True, random_state=RANDOM_STATE)

def make_pipeline(model, sampler=None):
    """sklearn Pipeline when no resampling (lighter serving deps), imblearn Pipeline otherwise."""
    if sampler is None:
        return Pipeline([("prep", build_preprocessor()), ("clf", model)])
    return ImbPipeline([("prep", build_preprocessor()), ("sampler", sampler), ("clf", model)])

def imbalance_setup(strategy, model_key, y):
    """Returns (sampler, extra model kwargs) for a strategy."""
    spw = float((y == 0).sum() / (y == 1).sum())
    if strategy == "class_weight":
        return None, ({"scale_pos_weight": spw} if model_key == "xgb" else {"class_weight": "balanced"})
    if strategy == "random_under":
        return RandomUnderSampler(sampling_strategy=SAMPLER_RATIO, random_state=RANDOM_STATE), {}
    if strategy == "random_over":
        return RandomOverSampler(sampling_strategy=SAMPLER_RATIO, random_state=RANDOM_STATE), {}
    if strategy == "smote":
        return SMOTE(sampling_strategy=SAMPLER_RATIO, random_state=RANDOM_STATE), {}
    return None, {}

tic("imbalance")
imb_path = ART / "imbalance_comparison.csv"
if RESUME and imb_path.exists():
    imb_df = pd.read_csv(imb_path)
else:
    n_imb = min(CFG["imb_sample"], len(X_dev))
    X_imb, _, y_imb, _ = train_test_split(X_dev, y_dev, train_size=n_imb, stratify=y_dev,
                                          random_state=RANDOM_STATE) if n_imb < len(X_dev) else (X_dev, None, y_dev, None)
    base_models = {"logreg": lambda kw: LogisticRegression(max_iter=1000, **kw),
                   "dtree": lambda kw: DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE, **kw)}
    rows = []
    for strategy in ["none", "class_weight", "random_under", "random_over", "smote"]:
        for mk, factory in base_models.items():
            sampler, kw = imbalance_setup(strategy, mk, y_imb)
            t0 = time.time()
            res = cross_validate(make_pipeline(factory(kw), sampler), X_imb, y_imb, cv=cv,
                                 scoring=SCORING, n_jobs=-1)
            row = {"strategy": strategy, "model": mk, "fit_seconds": round(time.time() - t0, 1)}
            for m in SCORING:
                row[f"{m}_mean"] = res[f"test_{m}"].mean(); row[f"{m}_std"] = res[f"test_{m}"].std()
            rows.append(row); print(row["strategy"], mk, f"F1={row['f1_mean']:.4f}  PR-AUC={row['pr_auc_mean']:.4f}")
    imb_df = pd.DataFrame(rows)
    imb_df.to_csv(imb_path, index=False)
display(imb_df.round(4))

ranking = imb_df.groupby("strategy")[f"{OBJECTIVE}_mean"].mean().sort_values(ascending=False)
IMB_STRATEGY = ranking.index[0]
print(f"Ranking (mean CV {OBJECTIVE}):\n", ranking.round(4), "\nChosen strategy:", IMB_STRATEGY)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=imb_df, x="strategy", y="f1_mean", hue="model", ax=ax)
ax.set_title("Imbalance strategies — CV F1"); ax.set_ylim(0, 1)
savefig("08_imbalance_strategies")
toc("imbalance")

## 6. Model training and hyperparameter tuning
Five models: the three required by the brief (Logistic Regression, Decision Tree, Random Forest) plus XGBoost and
HistGradientBoosting (the brief asks for five). `RandomizedSearchCV` with stratified k-fold, **refit on F1**
(PR-AUC, ROC-AUC, precision and recall are recorded too). Iteration counts come from the budget and are kept small
to limit run time; each finished search is checkpointed (the checkpoint name includes budget/strategy/objective,
so changing those settings never reuses stale results).

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

def model_specs(y):
    specs = {
        "logreg": (lambda kw: LogisticRegression(max_iter=2000, **kw),
                   {"clf__C": loguniform(1e-3, 1e2)}, -1),
        "dtree": (lambda kw: DecisionTreeClassifier(random_state=RANDOM_STATE, **kw),
                  {"clf__max_depth": randint(3, 20), "clf__min_samples_leaf": randint(1, 200),
                   "clf__criterion": ["gini", "entropy"]}, -1),
        "rf": (lambda kw: RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE, **kw),
               {"clf__n_estimators": randint(80, CFG["rf_max_trees"] + 1), "clf__max_depth": [10, 14, 18, 24],
                "clf__min_samples_leaf": randint(1, 20), "clf__max_features": ["sqrt", 0.5],
                "clf__max_samples": [0.5, 0.8]}, 1),
        "xgb": (lambda kw: XGBClassifier(tree_method="hist", n_jobs=-1, eval_metric="aucpr",
                                         random_state=RANDOM_STATE, **kw),
                {"clf__n_estimators": randint(100, 400), "clf__max_depth": randint(3, 9),
                 "clf__learning_rate": loguniform(0.02, 0.3), "clf__subsample": uniform(0.6, 0.4),
                 "clf__colsample_bytree": uniform(0.6, 0.4), "clf__min_child_weight": randint(1, 10)}, 1),
        "hgb": (lambda kw: HistGradientBoostingClassifier(random_state=RANDOM_STATE, **kw),
                {"clf__learning_rate": loguniform(0.03, 0.3), "clf__max_leaf_nodes": randint(15, 64),
                 "clf__min_samples_leaf": randint(20, 200), "clf__l2_regularization": loguniform(1e-4, 1)}, 1),
    }
    return specs

MODEL_NAMES = {"logreg": "Logistic Regression", "dtree": "Decision Tree", "rf": "Random Forest",
               "xgb": "XGBoost", "hgb": "HistGradientBoosting"}
MODELS_TO_RUN = ["logreg", "dtree", "rf", "xgb", "hgb"]

best_estimators, search_summary = {}, {}
for key, (factory, space, search_jobs) in model_specs(y_dev).items():
    if key not in MODELS_TO_RUN:
        continue
    ck = CKPT / f"{key}_{BUDGET}_{IMB_STRATEGY}_{OBJECTIVE}.joblib"
    if RESUME and ck.exists():
        saved = joblib.load(ck)
        best_estimators[key], search_summary[key] = saved["estimator"], saved["summary"]
        print(f"{key}: loaded checkpoint (CV F1={search_summary[key]['cv_f1_mean']:.4f})")
        continue
    sampler, kw = imbalance_setup(IMB_STRATEGY, key, y_dev)
    search = RandomizedSearchCV(make_pipeline(factory(kw), sampler), space, n_iter=CFG["n_iter"][key],
                                scoring=SCORING, refit=OBJECTIVE, cv=cv, n_jobs=search_jobs,
                                random_state=RANDOM_STATE, error_score="raise", verbose=1)
    t0 = time.time()
    search.fit(X_dev, y_dev)
    secs = round(time.time() - t0, 1)
    i = search.best_index_
    cvr = pd.DataFrame(search.cv_results_)
    cvr.to_csv(ART / "cv_results" / f"{key}.csv", index=False)
    summ = {"model": MODEL_NAMES[key], "search_seconds": secs, "n_iter": CFG["n_iter"][key],
            "best_params": search.best_params_, "fixed_params": kw,
            "sampler": type(sampler).__name__ if sampler is not None else None,
            **{f"cv_{m}_mean": float(cvr.loc[i, f"mean_test_{m}"]) for m in SCORING},
            **{f"cv_{m}_std": float(cvr.loc[i, f"std_test_{m}"]) for m in SCORING}}
    best_estimators[key], search_summary[key] = search.best_estimator_, summ
    joblib.dump({"estimator": search.best_estimator_, "summary": summ}, ck, compress=3)
    print(f"{key}: CV F1={summ['cv_f1_mean']:.4f} ± {summ['cv_f1_std']:.4f} | PR-AUC={summ['cv_pr_auc_mean']:.4f}  ({secs}s)\n  {search.best_params_}")

display(pd.DataFrame(search_summary).T[["cv_f1_mean", "cv_f1_std", "cv_precision_mean", "cv_recall_mean",
                                         "cv_pr_auc_mean", "search_seconds"]])

## 7. Evaluation on validation and test (true class distribution)
- **Threshold**: for each model, the F1-optimal threshold on the validation set (true class distribution).
  The minimum-cost threshold (`COST_FN × FN + COST_FP × FP`) is recorded for the business discussion.
- **Model selection**: highest validation F1 at its tuned threshold (ties → higher CV F1, then lower latency).
  `FINAL_MODEL` (Random Forest) is the deployed model; the criterion-based pick is recorded alongside it.
  The test set is only used to report final numbers.
- Single-row prediction latency is measured because the API scores transactions in real time.

In [ ]:
from sklearn.metrics import (accuracy_score, auc, average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve)

def choose_thresholds(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    prec, rec = prec[:-1], rec[:-1]
    P = y.sum()
    tp = rec * P
    fp = np.where(prec > 0, tp * (1 - prec) / np.maximum(prec, 1e-12), np.inf)
    fn = P - tp
    cost = COST_FN * fn + COST_FP * fp
    f1 = np.where(prec + rec > 0, 2 * prec * rec / (prec + rec), 0)
    return float(thr[np.argmin(cost)]), float(thr[np.argmax(f1)])

def metrics_at(y, p, t):
    yhat = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return {"threshold": t, "accuracy": accuracy_score(y, yhat), "precision": precision_score(y, yhat, zero_division=0),
            "recall": recall_score(y, yhat), "f1": f1_score(y, yhat), "roc_auc": roc_auc_score(y, p),
            "pr_auc": average_precision_score(y, p), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "expected_cost": int(COST_FN * fn + COST_FP * fp)}

def single_row_latency_ms(est, X, n=200):
    rows = X.sample(n, random_state=RANDOM_STATE)
    t0 = time.perf_counter()
    for i in range(n):
        est.predict_proba(rows.iloc[[i]])
    return (time.perf_counter() - t0) / n * 1000

tic("evaluate")
results, probs_val, probs_test = [], {}, {}
for key, est in best_estimators.items():
    pv = est.predict_proba(X_val)[:, 1]; pt = est.predict_proba(X_test)[:, 1]
    probs_val[key], probs_test[key] = pv.astype("float32"), pt.astype("float32")
    t_cost, t_f1 = choose_thresholds(y_val.values, pv)
    m_val = metrics_at(y_val.values, pv, t_f1)
    m_test_default = metrics_at(y_test.values, pt, 0.5)
    m_test = metrics_at(y_test.values, pt, t_f1)
    lat = single_row_latency_ms(est, X_test)
    search_summary[key].update({"threshold_cost": t_cost, "threshold_f1": t_f1, "latency_ms_single_row": lat,
                                "val_metrics": m_val, "test_metrics_at_threshold": m_test,
                                "test_metrics_at_0_5": m_test_default})
    row = {"model_key": key, "model": MODEL_NAMES[key], "cv_f1": search_summary[key]["cv_f1_mean"],
           "val_f1": m_val["f1"], "val_pr_auc": m_val["pr_auc"], "threshold": t_f1, "latency_ms": lat}
    row.update({f"test_{k}": v for k, v in m_test.items() if k != "threshold"})
    row.update({f"test@0.5_{k}": m_test_default[k] for k in ["precision", "recall", "f1"]})
    results.append(row)
    print(f"{key}: val F1={m_val['f1']:.4f} | test P={m_test['precision']:.3f} R={m_test['recall']:.3f} "
          f"F1={m_test['f1']:.3f} PR-AUC={m_test['pr_auc']:.4f} @t={t_f1:.3f} | {lat:.1f} ms/row")

comparison = pd.DataFrame(results).sort_values(["val_f1", "cv_f1", "latency_ms"], ascending=[False, False, True])
comparison.to_csv(ART / "model_comparison.csv", index=False)
display(comparison.round(4))
CRITERION_PICK = comparison.iloc[0]["model_key"]
BEST = FINAL_MODEL if FINAL_MODEL in best_estimators else CRITERION_PICK
print("Criterion-based pick (validation F1, ties -> CV F1):", MODEL_NAMES[CRITERION_PICK])
print("Final model:", MODEL_NAMES[BEST], "(chosen for deployment)" if BEST != CRITERION_PICK else "")
toc("evaluate")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for key in best_estimators:
    fpr, tpr, _ = roc_curve(y_test, probs_test[key])
    axes[0].plot(fpr, tpr, label=f"{MODEL_NAMES[key]} (AUC={auc(fpr, tpr):.4f})")
    pr, rc, _ = precision_recall_curve(y_test, probs_test[key])
    axes[1].plot(rc, pr, label=f"{MODEL_NAMES[key]} (AP={average_precision_score(y_test, probs_test[key]):.4f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC — test")
axes[1].axhline(y_test.mean(), color="k", ls="--", lw=0.8, label="baseline (prevalence)")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision–Recall — test")
for ax in axes: ax.legend(fontsize=8)
savefig("09_roc_pr_curves")

# Down-sampled curve points per model (Part 2 logs per-run ROC/PR plots to MLflow from these)
def _thin(*arrs, n=300):
    idx = np.unique(np.linspace(0, len(arrs[0]) - 1, min(n, len(arrs[0]))).astype(int))
    return [a[idx].round(6).tolist() for a in arrs]
curves = {}
for key in best_estimators:
    fpr, tpr, _ = roc_curve(y_test, probs_test[key])
    pr, rc, _ = precision_recall_curve(y_test, probs_test[key])
    fpr, tpr = _thin(fpr, tpr); pr, rc = _thin(pr, rc)
    curves[key] = {"roc": {"fpr": fpr, "tpr": tpr}, "pr": {"precision": pr, "recall": rc},
                   "prevalence": float(y_test.mean())}
save_json(curves, ART / "curves.json")

fig, axes = plt.subplots(1, len(best_estimators), figsize=(4 * len(best_estimators), 3.6))
for ax, key in zip(np.atleast_1d(axes), best_estimators):
    m = search_summary[key]["test_metrics_at_threshold"]
    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
    sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"])
    ax.set_title(f"{MODEL_NAMES[key]}\n t={m['threshold']:.3f}", fontsize=9)
savefig("10_confusion_matrices")

# Threshold trade-off for the best model (validation)
pr, rc, th = precision_recall_curve(y_val, probs_val[BEST])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(th, pr[:-1], label="precision"); ax.plot(th, rc[:-1], label="recall")
f1_curve = np.where(pr[:-1] + rc[:-1] > 0, 2 * pr[:-1] * rc[:-1] / (pr[:-1] + rc[:-1]), 0)
ax.plot(th, f1_curve, label="F1")
ax.axvline(search_summary[BEST]["threshold_f1"], color="k", ls="--", label="chosen (max F1)")
ax.axvline(0.5, color="grey", ls=":", label="0.5")
ax.set_xlabel("threshold"); ax.set_title(f"Threshold trade-off — {MODEL_NAMES[BEST]} (validation)"); ax.legend()
savefig("11_threshold_tradeoff")

## 8. Interpretation of the best model
Native importances (or coefficients), permutation importance on the raw API inputs, and SHAP.

In [ ]:
tic("interpret")
best = best_estimators[BEST]
prep, clf = best.named_steps["prep"], best.named_steps["clf"]
feat_names = list(prep.get_feature_names_out())
if hasattr(clf, "feature_importances_"):
    imp = pd.Series(clf.feature_importances_, index=feat_names, name="importance")
elif hasattr(clf, "coef_"):
    imp = pd.Series(np.abs(clf.coef_[0]), index=feat_names, name="importance")
else:
    imp = None

from sklearn.inspection import permutation_importance
Xs, _, ys, _ = train_test_split(X_val, y_val, train_size=min(30_000, len(X_val) - 1), stratify=y_val,
                                random_state=RANDOM_STATE)
perm = permutation_importance(best, Xs, ys, scoring="average_precision", n_repeats=3,  # threshold-free
                              random_state=RANDOM_STATE, n_jobs=1)
perm_s = pd.Series(perm.importances_mean, index=INPUT_FEATURES, name="perm_importance").sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
if imp is not None:
    imp.sort_values().tail(15).plot.barh(ax=axes[0], color="#4C72B0")
    axes[0].set_title(f"{MODEL_NAMES[BEST]} — model feature importance")
perm_s.sort_values().plot.barh(ax=axes[1], color="#55A868")
axes[1].set_title("Permutation importance on raw inputs (ΔPR-AUC, validation)")
savefig("12_feature_importance")

fi = pd.DataFrame({"feature": feat_names, "model_importance": imp.values if imp is not None else np.nan})
fi.sort_values("model_importance", ascending=False).to_csv(ART / "feature_importance.csv", index=False)
perm_s.to_frame().to_csv(ART / "permutation_importance.csv")

try:
    import shap
    n_shap = 300 if BEST == "rf" else 1000   # TreeExplainer on deep forests is slow
    Xt = prep.transform(Xs.sample(n_shap, random_state=RANDOM_STATE))
    Xt = pd.DataFrame(np.asarray(Xt), columns=feat_names)
    if BEST == "logreg":
        explainer = shap.LinearExplainer(clf, Xt)
    else:
        explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(Xt)
    if isinstance(sv, list):
        sv = sv[1]
    if np.ndim(sv) == 3:
        sv = sv[:, :, 1]
    shap.summary_plot(sv, Xt, show=False, max_display=15)
    plt.title(f"SHAP summary — {MODEL_NAMES[BEST]}")
    savefig("13_shap_summary")
    pd.Series(np.abs(sv).mean(0), index=feat_names, name="mean_abs_shap").sort_values(ascending=False) \
        .to_frame().to_csv(ART / "shap_importance.csv")
except Exception as e:  # SHAP is optional in the brief
    print("SHAP skipped:", repr(e))
toc("interpret")

## 9. Export: final pipeline, CI model, reload check and hand-off files for Part 2

In [ ]:
import sklearn
from sklearn.base import clone
tic("export")

# 9.1 Final pipeline (preprocessing + model in one object)
final_path = ART / "models" / "fraud_pipeline.pkl"
joblib.dump(best, final_path, compress=3)
reloaded = joblib.load(final_path)
chk = X_test.sample(5000, random_state=RANDOM_STATE)
p_orig, p_new = best.predict_proba(chk)[:, 1], reloaded.predict_proba(chk)[:, 1]
reload_check = {"rows_checked": len(chk), "max_abs_diff": float(np.max(np.abs(p_orig - p_new))),
                "predictions_identical": bool(np.array_equal(best.predict(chk), reloaded.predict(chk)))}
assert reload_check["predictions_identical"], "Reloaded pipeline predictions differ!"
print("Reload check:", reload_check, f"| size={final_path.stat().st_size / 1e6:.1f} MB")

# 9.2 Small CI model: same architecture & params, smaller ensemble, fitted on 20k rows (kept in git for tests)
ci_model = clone(best)
if BEST == "rf":
    ci_model.set_params(clf__n_estimators=30)
elif BEST == "xgb":
    ci_model.set_params(clf__n_estimators=50)
Xc, _, yc, _ = train_test_split(X_dev, y_dev, train_size=min(20_000, len(X_dev) // 2), stratify=y_dev, random_state=RANDOM_STATE)
ci_model.fit(Xc, yc)
ci_path = ART / "models" / "fraud_pipeline_ci.pkl"
joblib.dump(ci_model, ci_path, compress=3)
print(f"CI model size: {ci_path.stat().st_size / 1e6:.2f} MB")

# 9.3 Small labelled sample for tests / Azure smoke tests (true distribution + extra frauds)
ci_sample = pd.concat([test[test[TARGET] == 1].sample(min(100, int(y_test.sum())), random_state=RANDOM_STATE),
                       test[test[TARGET] == 0].sample(1900, random_state=RANDOM_STATE)])
ci_sample[INPUT_FEATURES + [TARGET]].to_csv(ART / "ci_sample.csv", index=False)

# 9.4 Example API requests with expected outputs
ex = pd.concat([test[test[TARGET] == 1].sample(3, random_state=1), test[test[TARGET] == 0].sample(3, random_state=1)])
ex_p = best.predict_proba(ex[INPUT_FEATURES])[:, 1]
thr = search_summary[BEST]["threshold_f1"]
sample_requests = [{"request": {k: (v.item() if hasattr(v, "item") else v) for k, v in r[INPUT_FEATURES].to_dict().items()},
                    "true_label": int(r[TARGET]), "expected_probability": float(p),
                    "expected_prediction": int(p >= thr)} for (_, r), p in zip(ex.iterrows(), ex_p)]
save_json(sample_requests, ART / "sample_requests.json")

# 9.5 Feature / schema spec (drives the Pydantic models in the API)
feature_spec = {
    "target": TARGET,
    "input_features": INPUT_FEATURES,
    "numeric_inputs": {c: {"dtype": "int" if c == "step" else "float", **input_ranges[c]} for c in RAW_NUMERIC},
    "categorical_inputs": {"type": {"allowed": TYPE_CATEGORIES}},
    "engineered_features": ENGINEERED_FEATURES,
    "model_feature_names": feat_names,
    "dropped_columns": {"nameOrig": "identifier", "nameDest": "identifier",
                        "isFlaggedFraud": "rule-based flag / leakage"},
    "notes": "All engineering lives in src/features.py and runs inside the pipeline; the API sends raw inputs only.",
}
save_json(feature_spec, ART / "feature_spec.json")

# 9.6 Best-model config — the single source of truth for Part 2 (retraining, MLflow, API threshold)
s = search_summary[BEST]
clf_params = {k.replace("clf__", ""): v for k, v in s["best_params"].items()}
best_model = {
    "model_key": BEST, "model_name": MODEL_NAMES[BEST],
    "estimator_class": f"{type(clf).__module__}.{type(clf).__name__}",
    "hyperparameters": {**clf_params, **s["fixed_params"]},
    "search_best_params_prefixed": s["best_params"],
    "imbalance": {"strategy": IMB_STRATEGY, "sampler": s["sampler"],
                  "sampler_ratio": SAMPLER_RATIO if s["sampler"] else None},
    "objective": OBJECTIVE,
    "decision_threshold": s["threshold_f1"], "threshold_min_cost": s["threshold_cost"],
    "threshold_policy": {"method": "max F1 on validation (true class distribution)",
                         "cost_reference": {"cost_fn": COST_FN, "cost_fp": COST_FP}},
    "selection_criterion": ("final model chosen for deployment; criterion = highest validation F1 at tuned "
                            "threshold, ties -> higher CV F1"),
    "criterion_pick": MODEL_NAMES[CRITERION_PICK], "final_model_override": FINAL_MODEL,
    "cv": {"folds": CFG["cv_folds"], "refit_scoring": OBJECTIVE,
           **{k: v for k, v in s.items() if k.startswith("cv_")}},
    "validation_metrics": s["val_metrics"], "test_metrics": s["test_metrics_at_threshold"],
    "test_metrics_at_0_5": s["test_metrics_at_0_5"], "latency_ms_single_row": s["latency_ms_single_row"],
    "training_data": split_info["dev_sample"], "random_state": RANDOM_STATE,
    "pipeline_file": "models/fraud_pipeline.pkl", "ci_pipeline_file": "models/fraud_pipeline_ci.pkl",
    "probability_note": ("Model trained on a negative-downsampled sample, so raw probabilities are inflated "
                         "relative to the true fraud rate; always apply decision_threshold (tuned on the "
                         "true-distribution validation set) rather than 0.5."),
}
save_json(best_model, ART / "best_model.json")
save_json(search_summary, ART / "all_models_summary.json")
save_json(reload_check, ART / "reload_check.json")

# 9.7 Environment versions — pin these in the VS Code requirements so the .pkl loads
import imblearn, xgboost
env_versions = {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                "imbalanced-learn": imblearn.__version__, "xgboost": xgboost.__version__,
                "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__,
                "scipy": __import__("scipy").__version__}
try:
    env_versions["shap"] = shap.__version__
except NameError:
    pass
save_json(env_versions, ART / "env_versions.json")
toc("export")

# 9.8 Run manifest
manifest = {"created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "notebook": "01_colab_model_selection.ipynb", "budget": BUDGET, "budget_config": CFG,
            "data_file": DATA_FILE.name, "rows_raw": clean_log["rows_raw"], "rows_clean": clean_log["rows_clean"],
            "split": split_info, "objective": OBJECTIVE, "imbalance_strategy": IMB_STRATEGY, "best_model": BEST,
            "timings_seconds": TIMINGS,
            "files": sorted(str(p.relative_to(ART)) for p in ART.rglob("*")
                            if p.is_file() and "checkpoints" not in p.parts)}
save_json(manifest, ART / "run_manifest.json")

In [ ]:
# 10. Summary + downloadable bundle (excludes checkpoints)
bundle_dir = Path("/content/artifacts_bundle")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
shutil.copytree(ART, bundle_dir, ignore=shutil.ignore_patterns("checkpoints"))
zip_path = shutil.make_archive(str(PROJECT_DIR / "artifacts_bundle"), "zip", bundle_dir)
print("Bundle:", zip_path)
print(f"\nBest model: {MODEL_NAMES[BEST]} | threshold={best_model['decision_threshold']:.4f}")
print("Test metrics:", {k: round(v, 4) if isinstance(v, float) else v for k, v in best_model["test_metrics"].items()})
print("Hyperparameters:", best_model["hyperparameters"])
print("Timings (s):", TIMINGS)